# Triple-barrier labels

This notebook labels OHLC bars with
[`TripleBarrierOHLC`](../functions_fin/TripleBarrierOHLC.md). Each bar is an
entry at its close. Over the next `h` bars its label is `+1` when the price
reaches the upper barrier first, `-1` when it reaches the lower barrier first,
and `0` when it reaches neither. The [labels guide](../labels.md) shows how to
pair a label with the features of its entry.

The bars are synthetic: each bar is a random walk of 12 steps, and the
volatility changes slowly over time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import (TripleBarrierOHLC, RollingMax, RollingMin, Lag, LogReturn, EwStd,
                      forecast_pairs)

# Each bar is a random walk of m log-price steps whose size follows a slowly varying
# volatility. The open is the previous close; the high and the low cover the open
# and the m steps.
rng = np.random.default_rng(24)
n, m = 4000, 12
log_vol = np.zeros(n)
for t in range(1, n):
    log_vol[t] = 0.995 * log_vol[t - 1] + 0.05 * rng.standard_normal()
step_size = 0.01 * np.exp(log_vol) / np.sqrt(m)
path = np.log(100) + np.cumsum(rng.standard_normal((n, m)) * step_size[:, None]).reshape(n, m)
prev_close = np.r_[np.log(100), path[:-1, -1]]
open_ = np.exp(prev_close)
high = np.exp(np.maximum(prev_close, path.max(axis=1)))
low = np.exp(np.minimum(prev_close, path.min(axis=1)))
close = np.exp(path[:, -1])

## Excursions

The barriers are set from how far the price runs within `h` bars. `RollingMax(h)`
of the high divided by the close `h` bars ago, from `Lag(h)`, is the highest
price over the last `h` bars relative to the close before them; `RollingMin(h)`
of the low gives the lowest. Paired with `count=h`, row `t` holds the highest
and the lowest price over the next `h` bars, relative to the close at `t`.

With `barrier_unit="vol"`, a barrier is a multiple of `s`, the volatility
expected over the horizon: the `EwStd(span=vol_span)` of the one-step log
returns of the close, times `sqrt(h)`. Dividing each excursion by `s` at its
entry puts it in the same unit.

In [ ]:
h, vol_span = 5, 100
s = EwStd(span=vol_span)(LogReturn(1)(close)) * np.sqrt(h)

start = Lag(h)(close)
run_up = RollingMax(h)(high) / start - 1
run_down = 1 - RollingMin(h)(low) / start
s_p, up_p = forecast_pairs(s, run_up, count=h, dropna=True)
_, down_p = forecast_pairs(s, run_down, count=h, dropna=True)
up_v, down_v = up_p / s_p, down_p / s_p

q = [0.25, 0.5, 0.75]
print("quantiles     25%    50%    75%")
print("run up    ", np.quantile(up_v, q).round(2))
print("run down  ", np.quantile(down_v, q).round(2))

## Choosing the barriers

A barrier at the median excursion is reached within `h` bars by half of the
entries. Here `upper` and `lower` are the two medians, rounded to one decimal.
The output at bar `t` is the entry at `t - h`, so `out[h:]` holds the entries
`0` to `n - h - 1` in order.

In [ ]:
upper = round(float(np.median(up_v)), 1)
lower = round(float(np.median(down_v)), 1)
out = TripleBarrierOHLC(horizon=h, upper=upper, lower=lower, barrier_unit="vol",
                        vol_span=vol_span)(open_, high, low, close)
label, ret, holding = out[h:, 0], out[h:, 1], out[h:, 2]   # row e: the entry at bar e
print(f"upper = {upper}, lower = {lower} expected deviations over {h} bars")

## Labels on the price chart

Each entry is marked at its close: green when the upper barrier is reached
first, red when the lower barrier is reached first, and grey when neither is
reached within `h` bars.

In [ ]:
window = slice(1000, 1150)
bars = np.arange(n)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(bars[window], close[window], color="#555555", lw=0.8)
for value, color in [(1, "#28a745"), (-1, "#c0392b"), (0, "#999999")]:
    at = bars[window][label[window] == value]
    ax.scatter(at, close[at], color=color, s=12, zorder=3, label=f"{value:+d}" if value else "0")
ax.set(title=f"triple-barrier labels, h = {h} bars", xlabel="bar", ylabel="close")
ax.legend(title="label", loc="upper left")
fig.tight_layout()

## Label counts

Each barrier on its own is reached within `h` bars by about half of the
entries. A label of `0` needs the price to stay between both barriers for all
`h` bars. `return` is the simple return from the entry close to the exit.

In [ ]:
values, counts = np.unique(label[~np.isnan(label)], return_counts=True)
for v, c in zip(values, counts):
    print(f"label {v:2.0f}: {c} entries")
print(f"mean return to exit: +1 {np.mean(ret[label == 1]):+.4f}, "
      f"-1 {np.mean(ret[label == -1]):+.4f}, 0 {np.mean(ret[label == 0]):+.4f}")

## Holding times

`holding` is the number of bars from the entry to the exit, from 1 to `h`. An
entry labeled `0` exits at the close of bar `e + h`, so its holding is `h`.

In [ ]:
edges = np.arange(0.5, h + 1.5)
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist([holding[label == 1], holding[label == -1], holding[label == 0]], bins=edges,
        stacked=True, color=["#28a745", "#c0392b", "#999999"], label=["+1", "-1", "0"])
ax.set(title="bars from entry to exit", xlabel="holding (bars)", ylabel="entries")
ax.legend(title="label")
fig.tight_layout()

## Ambiguous bars

When one bar's high reaches the upper barrier and its low reaches the lower
barrier, the order of the two within the bar is unknown, and the entry's
outputs are `NaN`. The count below is over the entries whose window is complete
and whose `s` is known, so every `NaN` among them comes from such a bar.

In [ ]:
known = ~np.isnan(s[:n - h])
ambiguous = known & np.isnan(label)
print(f"{ambiguous.sum()} ambiguous entries of {known.sum()}")

The narrower the barriers are relative to the range of one bar, the more
entries are ambiguous. The loop counts them for symmetric barriers from 0.25 to
2 expected deviations.

In [ ]:
for k in [0.25, 0.5, 1.0, 2.0]:
    lab = TripleBarrierOHLC(horizon=h, upper=k, lower=k, barrier_unit="vol",
                            vol_span=vol_span)(open_, high, low, close)[h:, 0]
    print(f"upper = lower = {k:4.2f}: {np.sum(known & np.isnan(lab)):4d} ambiguous entries")